# TACK inference (SD1.4)
Install `requirements.txt` first.
`model.py` defines the Bridge–CLIP–head; `pipeline.py` contains SD generation and the internal
latent img2img repair. Both branches share the same SD1.4 weights.

In [ ]:
from pathlib import Path
import json
import torch
from diffusers import DDIMScheduler, StableDiffusionPipeline
from IPython.display import display
from model import load_detector
from pipeline import TackPipeline

sd14 = Path("models/sd14")
model_dir = Path("models/tack-clip-vit-large-patch14")
clip_path = model_dir / "clip"
concept = "nudity"
output = Path("outputs/inference_single")
output.mkdir(parents=True, exist_ok=True)
seed = 42

## 1. Load SD1.4 and Bridge–CLIP–head

In [ ]:
pipe = TackPipeline.from_pretrained(
    sd14, torch_dtype=torch.float16, local_files_only=True, use_safetensors=True,
    safety_checker=None, requires_safety_checker=False,
).to("cuda")
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config, timestep_spacing="leading")
detector = load_detector(model_dir, concept, clip_path=clip_path, device="cuda")
original = StableDiffusionPipeline(**pipe.components, requires_safety_checker=False)
original.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)
original.set_progress_bar_config(disable=True)
sampling = dict(num_inference_steps=50, height=512, width=512, guidance_scale=7.5)
print(f"Loaded SD1.4, {model_dir.name}/ub.2.att.2_bridge.pt and {concept}_head.pt.")

## 2. Generate and repair
The repair branch uses positive prompt `""`, negative prompt `"nudity"`.
The original prompt resumes after repair.

In [ ]:
prompts = {
    "benign": "A ceramic mug on a wooden table, studio photograph",
    "art": "A classical marble statue of a nude adult woman in an art museum.",
}
runs = {}
for name, prompt in prompts.items():
    baseline = original(prompt, generator=torch.Generator("cuda").manual_seed(seed), **sampling).images[0]
    result = pipe(prompt, detector=detector, generator=torch.Generator("cuda").manual_seed(seed),
                  rewind_seed=43, **sampling).images[0]
    baseline.save(output / f"{name}_original.png")
    result.save(output / f"{name}_tack.png")
    runs[name] = dict(prompt=prompt, seed=seed, **pipe.last_run,
                      identical_to_original=baseline.tobytes() == result.tobytes())
    print(f"{name}: Original / TACK")
    display(baseline, result)
    print(json.dumps(pipe.last_run, indent=2))
(output / "run.json").write_text(json.dumps(runs, indent=2), encoding="utf-8")